Configuración y Carga de Datos

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import joblib
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

# 1. Buscador dinámico de la carpeta data
if os.path.exists('../data/dataset_ia_final.csv'):
    PATH_DATA = '../data'
elif os.path.exists('data/dataset_ia_final.csv'):
    PATH_DATA = 'data'
elif os.path.exists('dataset_ia_final.csv'):
    PATH_DATA = '.'
else:
    raise FileNotFoundError(" No encuentro el archivo 'dataset_ia_final.csv'.")

archivo = os.path.join(PATH_DATA, 'dataset_ia_final.csv')
print(f" Ruta de datos establecida: {PATH_DATA}")

print(" Cargando datos")
df = pd.read_csv(archivo, low_memory=False)
print(f"Datos cargados. Total: {len(df)} filas.")

📂 Ruta de datos establecida: ../data
⏳ Cargando datos en memoria...
✅ Datos cargados. Total inicial: 8215 filas.


Ingeniería de Características (Feature Engineering)

In [ ]:
print(" Procesando variables")

# 1. Baños (Mantenemos los decimales como pediste)
def extraer_banos(texto):
    if pd.isna(texto): return 1.0
    numeros = re.findall(r"[-+]?\d*\.\d+|\d+", str(texto))
    return float(numeros[0]) if numeros else 1.0
df['bathrooms_num'] = df['bathrooms_text'].apply(extraer_banos)

# 2. Habitaciones y Capacidad
df['bedrooms'] = df['bedrooms'].fillna(1.0)
df['accommodates'] = df['accommodates'].fillna(2.0)

# 3. Tipo de alojamiento
dict_room = {'Entire home/apt': 3, 'Private room': 2, 'Shared room': 1, 'Hotel room': 2}
df['room_type_num'] = df['room_type'].map(dict_room).fillna(3)

# 4. Confianza y Reputación (NUEVAS VARIABLES)
# Superhost: 't' -> 1, 'f' -> 0
df['host_is_superhost'] = df['host_is_superhost'].map({'t': 1, 'f': 0}).fillna(0)
# Número de reseñas (si es nulo, asumimos 0)
df['number_of_reviews'] = df['number_of_reviews'].fillna(0)

# 5. Extracción de Amenities Premium (NUEVAS VARIABLES)
# Pasamos todo a minúsculas para que sea fácil buscar
amenities_lower = df['amenities'].str.lower().fillna('')

# Creamos variables binarias (1 = Sí, 0 = No)
df['has_pool'] = amenities_lower.str.contains('pool').astype(int)
df['has_ac'] = amenities_lower.str.contains('air conditioning').astype(int)
df['has_parking'] = amenities_lower.str.contains('parking').astype(int)
df['has_elevator'] = amenities_lower.str.contains('elevator').astype(int)
df['has_balcony'] = amenities_lower.str.contains('balcony|patio').astype(int)
df['has_workspace'] = amenities_lower.str.contains('workspace').astype(int)

# 6. Rescate de la Renta Media
rentas_sevilla = {
    'Los Remedios': 30000.0, 'Nervión': 28000.0, 'Casco Antiguo': 26000.0,
    'Sur': 25000.0, 'Triana': 24000.0, 'San Pablo - Santa Justa': 22000.0,
    'Macarena': 20000.0, 'Bellavista - La Palmera': 21000.0,
    'Este - Alcosa - Torreblanca': 19000.0, 'Norte': 17000.0, 'Cerro - Amate': 16000.0
}
df['renta_media'] = df['neighbourhood_group_cleansed'].map(rentas_sevilla).fillna(22000.0)

# 7. Sentimiento
df['score_sentimiento'] = df['score_sentimiento'] if 'score_sentimiento' in df.columns else np.nan
df['score_sentimiento'] = df['score_sentimiento'].fillna(0.5)

# 8. Precios
if df['price'].dtype == 'object':
    df['price'] = df['price'].astype(str).str.replace(r'[^\d.]', '', regex=True)
    df['price'] = pd.to_numeric(df['price'], errors='coerce')

print(" Variables listas ")

⚙️ Procesando variables (Nivel PRO activado)...
✅ Variables listas (16 features preparadas).


Extracción de Diccionario para la Web

In [ ]:
print(" Generando diccionario de barrios")

# Agrupamos por barrio y sacamos la mediana de sus coordenadas y renta
df_barrios = df.groupby('neighbourhood_cleansed')[['latitude', 'longitude', 'renta_media']].median().reset_index()

# Lo convertimos en un diccionario de Python
dict_barrios = df_barrios.set_index('neighbourhood_cleansed').to_dict(orient='index')

# Lo guardamos en la carpeta data
ruta_dict = os.path.join(PATH_DATA, 'dict_barrios.pkl')
joblib.dump(dict_barrios, ruta_dict)

print(f" Diccionario guardado con {len(dict_barrios)} barrios en: {ruta_dict}")

🗺️ Generando diccionario de barrios para Streamlit...
✅ Diccionario guardado con 103 barrios en: ../data\dict_barrios.pkl


Entrenamiento y Guardado del Cerebro (KNN)

In [14]:
# Comprobar cuántos nulos hay en cada columna crítica
print("🔍 Diagnóstico de datos (Nulos por columna):")
cols_interes = ['price', 'latitude', 'longitude', 'accommodates', 'bedrooms', 'bathrooms_num', 'room_type_num', 'renta_media', 'score_sentimiento']
print(df[cols_interes].isna().sum())

🔍 Diagnóstico de datos (Nulos por columna):
price                634
latitude               0
longitude              0
accommodates           0
bedrooms               0
bathrooms_num          0
room_type_num          0
renta_media            0
score_sentimiento      0
dtype: int64


In [ ]:
print("Iniciando entrenamiento del modelo KNN")

# EL ORDEN DE ESTAS 16 VARIABLES ES MUY IMPORTANTE
features = [
    'latitude', 'longitude', 'accommodates', 'bedrooms', 'bathrooms_num', 
    'room_type_num', 'renta_media', 'score_sentimiento', 
    'host_is_superhost', 'number_of_reviews', 
    'has_pool', 'has_ac', 'has_parking', 'has_elevator', 'has_balcony', 'has_workspace'
]
target = 'price'

df_knn = df.dropna(subset=[target] + features).copy()

if len(df_knn) > 0:
    df_knn = df_knn[df_knn['price'] < 800] 

    X = df_knn[features]
    y = df_knn[target]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Damos el doble de peso al sentimiento (Índice 7) y al Superhost (Índice 8)
    X_scaled[:, 7] = X_scaled[:, 7] * 2.0 
    X_scaled[:, 8] = X_scaled[:, 8] * 2.0

    # Entrenamos buscando 7 vecinos
    knn_model = KNeighborsRegressor(n_neighbors=7, weights='distance')
    knn_model.fit(X_scaled, y)
    
    print(f" Modelo entrenado con {len(df_knn)} alojamientos.")

    joblib.dump(knn_model, os.path.join(PATH_DATA, 'knn_model.pkl'))
    joblib.dump(scaler, os.path.join(PATH_DATA, 'scaler.pkl'))
    print("modelo guardado.")
else:
    print("ERROR: 0 filas tras limpieza.")

🧠 Iniciando entrenamiento del modelo KNN PRO...
✅ ¡ÉXITO! Modelo entrenado con 7460 alojamientos.
💾 Cerebro IA guardado.
